<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/lab-08-compliance-assistant-for-meridian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 8 (graded) — Compliance assistant for Meridian
**Course 2: Generative AI and LLMs with Python — Chapter 8: RAG I: build a real pipeline**

**Problem brief (Leo Farkas, Meridian Bank):** "Our compliance team wastes hours searching
the rulebook. Build an assistant that answers policy questions with citations — and says
'I don't know' when the answer isn't in the documents."

**Data note:** Meridian's compliance rulebook is an internal document set (there is no
public URL to fetch a fictional bank's policy from) — the corpus below is a set of original,
realistic compliance documents authored for this course, covering real regulatory concepts
(KYC, AML, Reg E, overdraft policy) the way a real bank's internal RAG system would actually
index its own rulebook.

**What you'll submit:** the full pipeline, hitting the grounded-answer and citation targets,
and correct refusal on 10 out-of-corpus questions.

In [ ]:
!pip install -q sentence-transformers chromadb faiss-cpu

## 1. The corpus

In [ ]:
documents = [
    {'id': 'kyc-001', 'title': 'Customer Identification Program',
     'text': ('Meridian Bank requires a government-issued photo ID and proof of address for '
               'every new account opening, per the bank\'s Customer Identification Program (CIP). '
               'Accounts may not be activated for transactions until identity verification is '
               'complete. Enhanced due diligence applies to accounts flagged as higher-risk, '
               'including politically exposed persons and cash-intensive businesses.')},
    {'id': 'aml-002', 'title': 'Suspicious Activity Reporting',
     'text': ('Any transaction pattern inconsistent with a customer\'s known profile must be '
               'escalated to the AML team within 24 hours. A Suspicious Activity Report (SAR) is '
               'filed for transactions or patterns meeting regulatory thresholds, typically '
               'structuring deposits under $10,000 to avoid reporting requirements, or rapid '
               'movement of funds through multiple accounts with no clear business purpose.')},
    {'id': 'ege-003', 'title': 'Electronic Fund Transfer Error Resolution (Reg E)',
     'text': ('Customers have 60 days from the statement date to report an unauthorized '
               'electronic transfer. The bank must investigate and resolve the claim within 10 '
               'business days, or provisionally credit the account while investigating for up to '
               '45 days in complex cases. Customer liability is capped at $50 if reported within '
               '2 business days of discovering the unauthorized transfer.')},
    {'id': 'od-004', 'title': 'Overdraft Protection Policy',
     'text': ('Customers must affirmatively opt in to overdraft coverage for everyday debit card '
               'transactions and ATM withdrawals; without opt-in, such transactions are declined '
               'at no charge rather than triggering an overdraft fee. Overdraft fees are capped '
               'at 3 per day per account. Accounts overdrawn for more than 60 consecutive days '
               'are referred to collections.')},
    {'id': 'lend-005', 'title': 'Fair Lending & Adverse Action Notices',
     'text': ('Any credit application that is denied, or approved on materially less favorable '
               'terms than requested, requires an adverse action notice within 30 days stating '
               'the specific principal reasons for the decision. Race, color, religion, national '
               'origin, sex, marital status, and age may never be a factor in the credit decision.')},
    {'id': 'priv-006', 'title': 'Customer Data Privacy',
     'text': ('Customer financial data may be shared with third parties only for servicing the '
               'account, fraud prevention, or with explicit customer consent. Customers may opt '
               'out of data sharing for marketing purposes at any time via the privacy notice '
               'included with account opening and annually thereafter.')},
    {'id': 'card-007', 'title': 'Credit Card Dispute Process',
     'text': ('Cardholders have 60 days from the statement date to dispute a billing error in '
               'writing. The bank must acknowledge the dispute within 30 days and resolve it '
               'within two billing cycles, not exceeding 90 days. The disputed amount is not due '
               'while under investigation, and no late fee accrues on the disputed portion.')},
    {'id': 'branch-008', 'title': 'Branch Cash Handling Limits',
     'text': ('Teller cash drawers are limited to $10,000 at any time; amounts above this must be '
               'moved to the vault. Any single cash transaction of $10,000 or more requires a '
               'Currency Transaction Report (CTR) filed with FinCEN within 15 days.')},
]
print(f'{len(documents)} policy documents loaded.')

## 2. Chunk, embed, and index

In [ ]:
import re

def chunk_text(text, doc_id, title, max_sentences=2):
    sentences = re.split(r'(?<=[.!?]) +', text)
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk = ' '.join(sentences[i:i + max_sentences])
        chunks.append({'doc_id': doc_id, 'title': title, 'text': chunk})
    return chunks

all_chunks = []
for doc in documents:
    all_chunks.extend(chunk_text(doc['text'], doc['id'], doc['title']))
print(f'{len(all_chunks)} chunks from {len(documents)} documents.')

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')
chunk_texts = [c['text'] for c in all_chunks]
chunk_embeddings = embedder.encode(chunk_texts, normalize_embeddings=True)

try:
    import chromadb
    client = chromadb.Client()
    collection = client.get_or_create_collection('meridian_policy')
    collection.add(
        ids=[str(i) for i in range(len(all_chunks))],
        embeddings=chunk_embeddings.tolist(),
        documents=chunk_texts,
        metadatas=[{'doc_id': c['doc_id'], 'title': c['title']} for c in all_chunks],
    )
    VECTOR_BACKEND = 'chroma'
    print('Indexed with Chroma.')
except Exception as e:
    print(f'Chroma unavailable ({e}) — using the FAISS offline fallback.')
    import faiss
    index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
    index.add(chunk_embeddings.astype('float32'))
    VECTOR_BACKEND = 'faiss'

## 3. Retrieval

In [ ]:
def retrieve(query, top_k=3):
    q_emb = embedder.encode([query], normalize_embeddings=True)
    if VECTOR_BACKEND == 'chroma':
        res = collection.query(query_embeddings=q_emb.tolist(), n_results=top_k)
        return [{'text': d, 'title': m['title'], 'doc_id': m['doc_id'], 'score': 1 - dist}
                for d, m, dist in zip(res['documents'][0], res['metadatas'][0], res['distances'][0])]
    else:
        scores, idxs = index.search(q_emb.astype('float32'), top_k)
        return [{'text': all_chunks[i]['text'], 'title': all_chunks[i]['title'],
                  'doc_id': all_chunks[i]['doc_id'], 'score': float(s)}
                for s, i in zip(scores[0], idxs[0])]

for r in retrieve('How long do customers have to report an unauthorized transfer?'):
    print(f"[{r['score']:.3f}] {r['title']}: {r['text'][:100]}...")

## 4. Assemble the prompt + generate a cited, grounded answer

In [ ]:
import os
try:
    from google.colab import userdata
    API_KEY = userdata.get('LLM_API_KEY'); BASE_URL = userdata.get('LLM_BASE_URL')
except Exception:
    API_KEY = os.environ.get('LLM_API_KEY'); BASE_URL = os.environ.get('LLM_BASE_URL')
hosted_available = bool(API_KEY and BASE_URL)

if not hosted_available:
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    # newer transformers releases dropped the text2text-generation pipeline task and the
    # Text2TextGenerationPipeline class entirely, so we call generate() directly instead.
    _local_tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')
    _local_model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')

    def local_gen(prompt, max_new_tokens=100, **kwargs):
        ids = _local_tokenizer(prompt, return_tensors='pt', truncation=True).input_ids
        out = _local_model.generate(ids, max_new_tokens=max_new_tokens)
        return [{'generated_text': _local_tokenizer.decode(out[0], skip_special_tokens=True)}]

SYSTEM_PROMPT = (
    'Answer the question using ONLY the provided context. Cite the source title(s) you used '
    'in brackets. If the answer is not in the context, say exactly: '
    '"I don\'t know — this isn\'t covered in the policy documents I have access to."'
)

def answer_question(question, top_k=3, relevance_threshold=0.25):
    retrieved = retrieve(question, top_k)
    # a real grounding gate: if the best match is weak, refuse before even calling the model
    if not retrieved or retrieved[0]['score'] < relevance_threshold:
        return {"answer": "I don't know — this isn't covered in the policy documents I have access to.",
                'citations': []}

    context = '\n\n'.join(f"[{r['title']}] {r['text']}" for r in retrieved)
    prompt = f'{SYSTEM_PROMPT}\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:'

    if hosted_available:
        from openai import OpenAI
        client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
        resp = client.chat.completions.create(
            model='llama-3.1-8b-instant', messages=[{'role': 'user', 'content': prompt}], max_tokens=150,
        )
        answer = resp.choices[0].message.content
    else:
        answer = local_gen(prompt, max_new_tokens=100)[0]['generated_text']

    return {'answer': answer, 'citations': [r['title'] for r in retrieved]}

result = answer_question('How long do customers have to report an unauthorized electronic transfer?')
print('Answer:', result['answer'])
print('Citations:', result['citations'])

## 5. Evaluate: grounded-answer rate + correct refusal on 10 out-of-corpus questions

In [ ]:
in_corpus_questions = [
    'What ID is required to open an account?',
    'When must a Suspicious Activity Report be filed?',
    'What is the customer liability cap for an unauthorized transfer reported within 2 days?',
    'How many overdraft fees can be charged per day?',
    'What must an adverse action notice include?',
]
out_of_corpus_questions = [
    "What is Meridian Bank's mortgage interest rate?",
    'Does the bank offer cryptocurrency trading?',
    "What is the CEO's name?",
    'How many branches does the bank have?',
    'What is the routing number?',
    "What's the weather like at headquarters?",
    'Can I get a free toaster for opening an account?',
    'What programming language is the mobile app written in?',
    "What's the CEO's favorite color?",
    'Does the bank sponsor a sports team?',
]

grounded = sum(1 for q in in_corpus_questions if answer_question(q)['citations'])
correct_refusals = sum(1 for q in out_of_corpus_questions if "I don't know" in answer_question(q)['answer'])

print(f'Grounded-answer rate (in-corpus): {grounded}/{len(in_corpus_questions)}')
print(f'Correct refusals (out-of-corpus): {correct_refusals}/{len(out_of_corpus_questions)}')

## 6. Write-up (fill in)
Where did the pipeline get a refusal wrong — either refusing an in-corpus question or
answering an out-of-corpus one? What would you change about chunking, the relevance
threshold, or the prompt to fix it?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 8: RAG I*